# Representations, scaling, and the case for simple baselines

**Accompanies Section 6 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

The representation is the most consequential choice in an unsupervised pipeline, because
unsupervised methods have no mechanism for discovering that the representation was wrong. A
supervised model with a poor representation gives poor test predictions, which is informative.
An unsupervised method with a poor representation gives clean, confident, meaningless structure.

### Learning objectives
- Compute three families of descriptor for the same molecules and compare the similarity structures they induce
- See that "similar" is a property of the representation, not of the molecules
- Understand why binary fingerprints should not be standardized and fed to Euclidean methods
- Establish a simple baseline that anything more elaborate has to beat

### What this notebook is designed to make go wrong
Two descriptor sets that disagree about which molecules are neighbors, so a clustering built on
either one tells a different story with equal confidence.

### What you need installed
RDKit, scikit-learn, SciPy, pandas, NumPy and matplotlib, plus ASE, which the
manuscript-figure cell at the end uses to read the QM7 structures. The `environment.yml` at the
repository root installs them; nothing optional.

### Roughly how long it takes
A few minutes, most of it building representations for 1,000 molecules.

In [ ]:
# CANONICAL SETUP CELL. Paste verbatim as the first code cell of every notebook.
# Import lines for libraries a given notebook does not use may be dropped, and
# lines it needs extra (pandas, rdkit, sklearn) may be added, but the style
# block, SEED and DATA must be identical everywhere.
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. Build three representations of the same molecules

### Load the sample, and standardize it before anything else

We take the first 1,000 records of the ZINC-250k sample, which carry logP, QED and a synthetic
accessibility score. The SMILES are as distributed. ZINC-250k was cleaned upstream, so this
file holds no salts and no repeated strings, and what the pipeline below actually changes is
the formal charge that a substantial minority of the records carry. A table assembled from
several sources would give it more to do.

That matters more here than in a supervised setting. Two records for one compound that differ
only in how they were written land in different regions of descriptor space, so they can end up
in different clusters and get reported as a chemical distinction. Standardization is the fix,
and it is a set of chemical judgments rather than a formality, so keep the steps short and
report them.

In [ ]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")   # parse failures are counted below, not printed

# The shared sample holds 10,000 records. The comparisons below are pairwise, so
# take the first 1,000 rows and keep that quadratic work small.
zinc = pd.read_csv(DATA / "zinc-250k-sample.csv").head(1000)
print(f"{len(zinc)} records, columns: {list(zinc.columns)}")

# Build the RDKit helpers once and reuse them. Constructing them per molecule
# dominates the runtime on anything larger than this sample.
largest_fragment = rdMolStandardize.LargestFragmentChooser()
uncharger = rdMolStandardize.Uncharger()


def standardize(smiles):
    """Bring one SMILES to a single convention, or return None if it will not parse.

    Parse and sanitize, keep the largest fragment, which strips salts and
    counter-ions, neutralize the formal charges that fragment removal leaves
    behind, then write a canonical SMILES. Two records describing the same
    compound come out as the same string.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = largest_fragment.choose(mol)
    mol = uncharger.uncharge(mol)
    return Chem.MolToSmiles(mol, canonical=True)


cleaned = [standardize(s.strip()) for s in zinc["smiles"]]
n_failed = sum(s is None for s in cleaned)
smiles = sorted({s for s in cleaned if s is not None})
print(f"{n_failed} failed to parse, {len(smiles)} unique structures after standardization")

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Descriptors import (
    BertzCT, FractionCSP3, MolLogP, MolWt, NumHAcceptors, NumHDonors,
    NumRotatableBonds, TPSA,
)

mols = [Chem.MolFromSmiles(s) for s in smiles]
print(f"{len(mols)} standardized molecules")

# (a) Morgan / ECFP fingerprints: substructure presence
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
fps_bit = [morgan.GetFingerprint(m) for m in mols]
fps_np = np.array([morgan.GetFingerprintAsNumPy(m) for m in mols])

# (b) MACCS keys: 166 predefined structural fragments
from rdkit.Chem import MACCSkeys
maccs = np.array([np.array(MACCSkeys.GenMACCSKeys(m)) for m in mols])

# (c) Physicochemical descriptors: heterogeneous units and scales
DESCRIPTORS = [MolWt, MolLogP, TPSA, NumHDonors, NumHAcceptors,
               NumRotatableBonds, FractionCSP3, BertzCT]
physchem = np.array([[f(m) for f in DESCRIPTORS] for m in mols])

print(f"Morgan     {fps_np.shape}")
print(f"MACCS      {maccs.shape}")
print(f"physchem   {physchem.shape}")

### See the molecules the representations describe

Everything that follows turns these structures into vectors and compares the vectors. It is worth a look at the molecules first, so that a later claim about which of them are neighbors has something concrete to be checked against.

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display


def draw_molecules(smiles_list, legends=None, n=8, per_row=4):
    """Grid depiction of the first n molecules, for orientation rather than analysis.

    Molecules that fail to parse are dropped along with their legend, so a single
    bad string does not blank the whole grid.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list[:n]]
    legs = list(legends[:n]) if legends is not None else None
    keep = [i for i, m in enumerate(mols) if m is not None]
    mols = [mols[i] for i in keep]
    legs = [legs[i] for i in keep] if legs is not None else None
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=(260, 200), legends=legs)


# Eight of the molecules every representation below describes, labeled by weight.
display(draw_molecules(smiles, legends=[f"{MolWt(m):.0f} Da" for m in mols], n=8))

### Compare the descriptor families side by side

The article's descriptor table tells you what each family *assumes*. This cell
shows what those assumptions do to the numbers, on the same molecules. Look at
the spread of pairwise distances: a representation whose distances are all
nearly equal has no neighborhood structure left for a clustering or an
embedding to find.

In [ ]:
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler

# Standardized here and not reused from the next section, so this cell
# stands on its own and the comparison is like for like.
physchem_std = StandardScaler().fit_transform(physchem)

def profile(name, M, metric="euclidean"):
    d = pdist(M, metric=metric)
    sparsity = float((M == 0).mean())
    # relative spread of pairwise distances: near zero means distances have
    # concentrated and "nearest neighbor" has stopped discriminating
    return {
        "family": name, "shape": M.shape, "sparsity": sparsity,
        "metric": metric, "d_mean": d.mean(), "d_spread": d.std() / d.mean(),
    }

rows = [
    profile("Morgan (binary)", fps_np, metric="jaccard"),
    profile("MACCS keys", maccs, metric="jaccard"),
    profile("physicochemical (raw)", physchem),
    profile("physicochemical (standardized)", physchem_std),
]

print(f"{'family':<32s} {'shape':>12s} {'sparsity':>9s} {'metric':>10s} {'d spread':>9s}")
for r in rows:
    print(f"{r['family']:<32s} {str(r['shape']):>12s} {r['sparsity']:9.2f} "
          f"{r['metric']:>10s} {r['d_spread']:9.3f}")
print("\nd spread = std/mean of the pairwise distances. Small values mean the")
print("distances have concentrated: everything is roughly equidistant from")
print("everything else, and neighborhood-based methods have nothing to work with.")

## 2. Heterogeneous scales make standardization mandatory

Look at the ranges below. Molecular weight spans hundreds; the number of hydrogen bond donors
spans single digits. In a Euclidean distance, molecular weight would drown out everything else,
not because it is chemically more important, but because its numbers are bigger.

In [ ]:
names = [f.__name__ for f in DESCRIPTORS]
print(f"{'descriptor':<20s} {'min':>10s} {'max':>10s} {'std':>10s}")
for i, name in enumerate(names):
    col = physchem[:, i]
    print(f"{name:<20s} {col.min():10.2f} {col.max():10.2f} {col.std():10.2f}")

from sklearn.preprocessing import StandardScaler
physchem_scaled = StandardScaler().fit_transform(physchem)
print("\nAfter standardization every descriptor has unit variance and contributes equally.")

### Decide which descriptors are earning their place

Keeping every descriptor by default is a choice, and a lazy one when several carry almost the same information. A supervised diagnostic, borrowed to inform an unsupervised choice, gives a cheap read on it: fit a random forest to predict a held-out property from the descriptors and look at which ones it leans on. Two cautions keep this honest. Importance from a correlated feature set is not a clean attribution, because when two descriptors say the same thing the forest can split the credit between them arbitrarily. And the useful reading is negative rather than positive: the descriptors near zero are contributing nothing the others do not already supply and are candidates to drop, which is a safer conclusion than ranking the ones at the top.

In [ ]:
from rdkit.Chem import QED
from sklearn.ensemble import RandomForestRegressor

# Names in the order of DESCRIPTORS; the functions' own __name__ is "<lambda>"
# for most RDKit descriptors, so spell them out.
DESC_NAMES = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
              "NumRotatableBonds", "FractionCSP3", "BertzCT"]

# QED is a composite drug-likeness score. It is derived in part from these same
# physicochemical properties, so this is not a fully external test, but it avoids
# the outright circularity of predicting a descriptor that is itself a column, and
# it is enough to reveal which columns the forest never uses.
target = np.array([QED.qed(m) for m in mols])
forest = RandomForestRegressor(n_estimators=200, random_state=SEED).fit(physchem_scaled, target)

print("Descriptor importance for predicting QED (higher = used more by the forest):")
for i in np.argsort(forest.feature_importances_)[::-1]:
    print(f"  {DESC_NAMES[i]:<18s} {forest.feature_importances_[i]:.3f}")

## 3. Do the representations agree about who is similar to whom?

For a handful of query molecules, find the ten nearest neighbors under each representation
and measure the overlap: the fraction of a point's k nearest neighbors that two versions of
the data agree on. Work out chance before you read the number. Drawing k neighbors at random
from n points gives an overlap of about k/n, so with 10 neighbors among a thousand molecules
chance is around 0.01. If the representations agreed, the overlap would be near one.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def knn_sets(matrix, metric, k=10):
    nn = NearestNeighbors(n_neighbors=k + 1, metric=metric).fit(matrix)
    return nn.kneighbors(matrix, return_distance=False)[:, 1:]

# Jaccard for binary fingerprints, Euclidean for standardized real-valued descriptors.
nbrs_morgan = knn_sets(fps_np, "jaccard")
nbrs_maccs = knn_sets(maccs, "jaccard")
nbrs_physchem = knn_sets(physchem_scaled, "euclidean")

def mean_overlap(a, b, k=10):
    return np.mean([np.intersect1d(a[i], b[i]).size / k for i in range(len(a))])

print("Mean overlap of the 10 nearest neighbors between representations:")
print(f"  Morgan   vs MACCS    : {mean_overlap(nbrs_morgan, nbrs_maccs):.3f}")
print(f"  Morgan   vs physchem : {mean_overlap(nbrs_morgan, nbrs_physchem):.3f}")
print(f"  MACCS    vs physchem : {mean_overlap(nbrs_maccs, nbrs_physchem):.3f}")
print()
print("These are the SAME molecules. Any clustering built on one representation")
print("would disagree with a clustering built on another, with equal confidence.")

## 4. Use Tanimoto on binary fingerprints, not Euclidean distance

Standardizing a binary fingerprint and passing it to a Euclidean method inflates the influence of
rare bits: a bit set in three molecules out of a thousand gets a very large standardized value.
Tanimoto/Jaccard similarity is the right choice, and the difference is not subtle.

In [ ]:
query = 0
tanimoto = np.array(DataStructs.BulkTanimotoSimilarity(fps_bit[query], fps_bit))

from scipy.spatial.distance import cdist
fps_std = StandardScaler().fit_transform(fps_np.astype(float))
euclid_std = cdist(fps_std[query:query + 1], fps_std, metric="euclidean").ravel()

order_tanimoto = np.argsort(-tanimoto)[1:6]
order_euclid = np.argsort(euclid_std)[1:6]

print(f"Query: {smiles[query]}\n")
print("Top 5 neighbors by Tanimoto (correct):")
for i in order_tanimoto:
    print(f"   T={tanimoto[i]:.3f}  {smiles[i]}")
print("\nTop 5 neighbors by Euclidean distance on standardized bits (wrong):")
for i in order_euclid:
    print(f"   T={tanimoto[i]:.3f}  {smiles[i]}")
print(f"\nOverlap between the two lists: {len(set(order_tanimoto) & set(order_euclid))}/5")

### Check the two metric properties the table cannot show you

Table 3 in the article says Tanimoto "falls systematically with molecular size"
and that cosine is "blind to magnitude". Both take one line of code to check.

**Tanimoto and size.** Larger molecules set more bits, so the union grows faster
than the intersection and similarity falls, independently of chemistry. In a
clustering that bias is not neutral: it produces groups organized partly by
size. Check for it by asking whether cluster membership predicts heavy-atom
count.

**Cosine and magnitude.** Cosine measures the angle only, which is right for
embeddings whose norm is an artifact of token count and catastrophic for
spectra, where intensity is the signal.

In [ ]:
# --- Tanimoto falls with molecular size, independently of chemistry ---
heavy = np.array([m.GetNumHeavyAtoms() for m in mols])
sim = np.zeros((len(mols), len(mols)))
for i in range(len(mols)):
    sim[i] = DataStructs.BulkTanimotoSimilarity(fps_bit[i], fps_bit)
np.fill_diagonal(sim, np.nan)
mean_sim = np.nanmean(sim, axis=1)

r = np.corrcoef(heavy, mean_sim)[0, 1]
print("Tanimoto similarity versus molecular size")
print(f"  heavy atoms range {heavy.min()}-{heavy.max()}")
print(f"  correlation of heavy-atom count with mean Tanimoto: r = {r:+.3f}")
for lo, hi in [(0, 20), (20, 26), (26, 100)]:
    m = (heavy >= lo) & (heavy < hi)
    if m.sum():
        print(f"    {lo:>2d}-{hi if hi < 100 else '+':>3} heavy atoms  n={m.sum():4d}  "
              f"mean similarity {mean_sim[m].mean():.3f}")
print("  A cluster of large molecules is not a chemical family until this is ruled out.\n")

# --- cosine discards magnitude: right for embeddings, wrong for spectra ---
grid = np.arange(200)
shape = np.exp(-((grid - 80) ** 2) / 200) + 0.3 * np.exp(-((grid - 140) ** 2) / 400)
weak, strong = shape, 10.0 * shape          # same spectrum, tenfold intensity


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Cosine versus magnitude, on two spectra of identical shape")
print(f"  cosine similarity   {cosine(weak, strong):.6f}   <- identical")
print(f"  Euclidean distance  {np.linalg.norm(weak - strong):.2f}      <- not identical")
print("  Use cosine where the norm is an artifact; never where it is the measurement.")

## 5. Beat PCA plus k-means before you claim anything more elaborate

The manuscript states this as strongly as it can: **PCA on standardized, well-chosen descriptors
followed by k-means is the reference** that anything more elaborate has to justify itself against.
Below, we build that reference, put a random projection underneath it as a floor, and put one more
elaborate method (t-SNE) on top, so it is visible what "beating the baseline" does and does not
establish.

If an elaborate pipeline does not beat this, it is not justified, and "beat" means beat
*significantly* (notebook 09), not beat numerically. It also has to beat the baseline on a metric
the elaborate method did not itself optimize: a neighbor embedding wins trustworthiness and
neighbor preservation almost by construction, so those wins alone are close to a tautology, and
notebook 07 adds the property-retention score that breaks it.

### Score the embedding, then test whether there are clusters at all

A two-dimensional embedding is a model of the data, so report it with numbers and not only as
a picture. Three cheap metrics cover most of what you need. **Trustworthiness** asks whether the
embedding invented neighbors: points that sit together in the picture but were far apart in the
input space. **Continuity** asks the opposite, whether it lost neighbors that were close to begin
with, and it is the same computation with the two spaces swapped. Both run from 0 to 1 with no
absolute threshold, so read them against the same measurement on a method you already trust, and
read them as a pair: they trade off against each other, and either one on its own hides half the
story. **Neighbor preservation** counts the fraction of each point's k nearest neighbors that
survived, and it is the easiest of the three to explain to someone else. Section 3's chance
baseline of about k/n applies to it, so work that out before a number impresses you.

The clusterability test is a different question. The **silhouette score** measures, for each
point, how much closer it sits to its own cluster than to the nearest other cluster, averaged
over all points and running from -1 to +1. Higher looks better, and that is the trap: k-means
optimizes something very close to the silhouette, so a high value can mean no more than that the
algorithm did its job. What makes it readable is a **permutation null**, which shuffles each
feature column independently, leaving every feature's own distribution untouched and destroying
the joint structure that clusters live in. Score a few dozen shuffled copies and you have the
silhouettes that structureless data with these same marginals produces. Run this before you
choose a number of clusters. If the observed silhouette sits inside that distribution, there is
nothing there, and no choice of k repairs the analysis.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import trustworthiness
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.random_projection import GaussianRandomProjection

# scikit-learn already ships trustworthiness, so we import it instead of
# rewriting it, and get continuity by calling it with the spaces swapped.


def continuity(X, embedding, n_neighbors=12):
    """Neighbors the embedding lost, measured as trustworthiness in reverse."""
    return float(trustworthiness(embedding, X, n_neighbors=n_neighbors))


def knn_indices(A, n_neighbors):
    """Each point's k nearest neighbors, with the point itself dropped."""
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(A)
    return nn.kneighbors(A, return_distance=False)[:, 1:]


def neighbor_preservation(X, embedding, n_neighbors=12):
    """Mean fraction of each point's k nearest neighbors that survive the embedding."""
    before, after = knn_indices(X, n_neighbors), knn_indices(embedding, n_neighbors)
    return float(np.mean([
        np.intersect1d(before[i], after[i]).size / n_neighbors
        for i in range(len(before))
    ]))


def evaluate_embeddings(X, embeddings, n_neighbors=12, random_state=0):
    """Score several embeddings side by side, with a random projection for scale.

    Keep the random projection. It costs nothing, it comes with distortion
    guarantees (Johnson-Lindenstrauss), and it separates how much of an
    embedding's apparent quality is the method from how much is the data being
    easy to embed.
    """
    embeddings = dict(embeddings)
    embeddings["random projection"] = GaussianRandomProjection(
        n_components=2, random_state=random_state).fit_transform(X)
    rows = [
        {
            "method": name,
            "trustworthiness": float(trustworthiness(X, emb, n_neighbors=n_neighbors)),
            "continuity": continuity(X, emb, n_neighbors),
            f"{n_neighbors}nn_preserved": neighbor_preservation(X, emb, n_neighbors),
        }
        for name, emb in embeddings.items()
    ]
    return (pd.DataFrame(rows)
            .sort_values("trustworthiness", ascending=False)
            .reset_index(drop=True))


def permutation_null(X, rng):
    """Shuffle each column on its own: the marginals survive, the joint structure does not."""
    shuffled = np.array(X, dtype=float)
    for j in range(shuffled.shape[1]):
        rng.shuffle(shuffled[:, j])
    return shuffled


def clusterability(X, n_clusters=4, n_replicates=49, random_state=0):
    """Compare the silhouette of a k-means fit against a permutation null.

    Returns the observed silhouette, the null values, and a one-sided empirical
    p-value from the (r + 1) / (n + 1) estimator, which never reports zero: with
    49 replicates the smallest value it can return is 0.02, which is honest
    about the resolution of the test.
    """
    def silhouette_of(data):
        labels = KMeans(n_clusters=n_clusters, n_init=10,
                        random_state=random_state).fit_predict(data)
        return float(silhouette_score(data, labels))

    null_rng = np.random.default_rng(random_state)
    observed = silhouette_of(X)
    null = np.array([silhouette_of(permutation_null(X, null_rng))
                     for _ in range(n_replicates)])
    p_value = (int(np.sum(null >= observed)) + 1) / (n_replicates + 1)
    return observed, null, p_value

### Read the p-value as detection, not as effect size

The **p-value** below is the fraction of null replicates that scored at least as high as the
real data. Small means the silhouette is not something structureless data produces easily. It
says nothing about how large the effect is, or whether it is chemically interesting.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

baseline = PCA(n_components=2, random_state=SEED).fit_transform(physchem_scaled)
# One method more elaborate than the baseline, run through the same yardstick.
elaborate = TSNE(n_components=2, perplexity=30, init="pca",
                 random_state=SEED, max_iter=1000).fit_transform(physchem_scaled)

print("Is this descriptor space even clusterable?")
observed, null, p_value = clusterability(physchem_scaled, n_clusters=4,
                                         n_replicates=49, random_state=SEED)
lo, hi = np.percentile(null, [2.5, 97.5])
effect = (observed - null.mean()) / null.std(ddof=1)
print(f"  observed silhouette  {observed:.4f}")
print(f"  permutation null     {null.mean():.4f} [{lo:.4f}, {hi:.4f}]")
print(f"  p = {p_value:.4f}, {effect:+.1f} SD from the null")

print("\nDoes the more elaborate method beat the baseline, and the baseline the floor?")
quality = evaluate_embeddings(
    physchem_scaled,
    {"PCA (baseline)": baseline, "t-SNE (elaborate)": elaborate},
    n_neighbors=12, random_state=SEED)
print(quality.round(3).to_string(index=False))

Read the table against the floor first, then against the baseline. Both real methods clear the
random projection, so neither is merely benefiting from a reduction in feature count. t-SNE then
beats PCA on trustworthiness and on neighbor preservation and loses a little continuity to it.
That split is the whole point: t-SNE optimizes local neighborhoods, so it wins the metrics that
score local neighborhoods almost by construction, which is not on its own a reason to prefer it.
The manuscript's question is whether it beats the baseline on something it did *not* optimize, and
by enough to matter: notebook 07 adds property retention for exactly that, and notebook 09 supplies
the significance test that separates "beats numerically" from "beats defensibly." Here, on eight
interpretable descriptors that PCA already renders faithfully, the elaborate method buys a
neighbor-preservation gain at the cost of axes a chemist can no longer name.

## 6. Missing values decide things too

Descriptor tables assembled from more than one source arrive with gaps, and most
implementations either fail on `NaN` or propagate it silently into a distance. The checklist
item is short (*diagnose missingness before computing distances*), and it is there because
**descriptors are rarely missing at random**. A property that could not be
computed usually could not be computed for a *reason*, and the reason is often chemical.

The sample below has no gaps, so we make some the way real data does: not uniformly, but
concentrated in the molecules at one end of a property range. Then we compare the two usual
treatments, dropping the incomplete rows or imputing, and ask what the checklist asks: does the
choice change the downstream structure?

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors

rng_missing = np.random.default_rng(SEED)

# Missing NOT at random: the heaviest quarter of the sample loses its logP,
# which is what happens when a property is hard to measure at one extreme.
mw_col = names.index("MolWt") if "MolWt" in names else 0
logp_col = names.index("MolLogP") if "MolLogP" in names else 1
heavy = physchem[:, mw_col] > np.quantile(physchem[:, mw_col], 0.75)

with_gaps = physchem.copy()
with_gaps[heavy, logp_col] = np.nan
print(f"{heavy.sum()} of {len(physchem)} molecules lost their {names[logp_col]} "
      f"({100 * heavy.mean():.0f}% of rows, 1 of {physchem.shape[1]} columns)")

# Treatment A: drop incomplete rows. Note what leaves with them.
kept = ~np.isnan(with_gaps).any(axis=1)
print(f"\ndeletion keeps {kept.sum()} rows and removes {(~kept).sum()}")
print(f"  mean {names[mw_col]} of the rows kept   : {physchem[kept, mw_col].mean():8.1f}")
print(f"  mean {names[mw_col]} of the rows removed: {physchem[~kept, mw_col].mean():8.1f}")
print("  Deletion did not remove a random quarter of the data. It removed the heavy end.")

# Treatment B: impute, then ask whether the neighbor structure moved.
imputed = SimpleImputer(strategy="median").fit_transform(with_gaps)
scaler = StandardScaler()


def knn_sets(matrix, k=12):
    Z = scaler.fit_transform(matrix)
    idx = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z, return_distance=False)
    return idx[:, 1:]


truth = knn_sets(physchem)
after = knn_sets(imputed)
overlap = np.mean([np.intersect1d(a, b).size / truth.shape[1]
                   for a, b in zip(truth, after)])
print(f"\n12-NN overlap between the complete data and the median-imputed data: {overlap:.3f}")
print("Report this number. An imputation that moves the neighbor structure has")
print("changed the analysis, and the fitted imputer belongs inside the split (Section 5).")

## 7. Remember that a SMILES carries no coordinates

The descriptors so far read a molecular graph, which a SMILES string supplies directly. Anything geometric needs 3D coordinates, and those have to be generated. Three points are easy to miss. Hydrogens have to be added explicitly before embedding, because a SMILES leaves them implicit and their positions matter for any geometric descriptor. ETKDG is a distance-geometry method seeded with real torsion preferences, so it is a sampler rather than a deterministic function: a different seed gives a different conformer, and the call below fixes the seed so the notebook reproduces. And a single conformer is one draw from a distribution, so a descriptor computed from one embedding inherits that arbitrariness, which is the reason QM7 is distributed with coordinates rather than as SMILES.

In [ ]:
from rdkit.Chem import AllChem

mol3d = Chem.AddHs(Chem.MolFromSmiles(smiles[0]))
params = AllChem.ETKDGv3()
params.randomSeed = SEED                      # a sampler, not a function: fix the seed
AllChem.EmbedMolecule(mol3d, params)
AllChem.MMFFOptimizeMolecule(mol3d)

flat = Chem.MolFromSmiles(smiles[0])
print(f"{smiles[0]}")
print(f"  heavy atoms in the SMILES graph:        {flat.GetNumAtoms()}")
print(f"  atoms after adding explicit hydrogens:  {mol3d.GetNumAtoms()}")
print(f"  conformers embedded:                    {mol3d.GetNumConformers()}")

## 8. Describe local atomic environments with SOAP, and score it against a baseline

Every representation so far has been whole-molecule. SOAP is the standard *local* one: around each atom it expands the density of neighboring atoms in radial basis functions and spherical harmonics, and contracts that into a power spectrum that is invariant to rotation, translation and permutation of identical atoms. That gives a vector per atom. A molecule-level vector is then the average of the per-atom vectors, and the averaging is a real choice, because it discards which environments were present in what proportion: two molecules with different environments but the same mean become indistinguishable.

SOAP needs 3D coordinates, so QM7 is the natural data set here, and it ships with them. To keep the descriptor honest we do not read a validity index off it; we give it a job with a right answer, predicting atomization energy, and score it on molecules it never saw against the trivial baseline of predicting the training mean.

In [ ]:
from ase.io import read
from dscribe.descriptors import SOAP
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

BOHR = 0.529177210903      # qm7.xyz positions are in bohr; convert for a cutoff in angstrom
qm7 = read(str(DATA / "qm7.xyz"), index=":")
subset = [qm7[i] for i in np.random.default_rng(SEED).choice(len(qm7), 1000, replace=False)]
for atoms in subset:
    atoms.set_positions(atoms.get_positions() * BOHR)
energies = np.array([a.info["atomization_energy"] for a in subset])

# r_cut is in angstrom, because the positions above were converted to angstrom.
# 4.0 angstrom reaches a little beyond the first coordination shell. average="outer"
# forms the rotation-invariant power spectrum for each atom first and then averages
# those per-atom spectra into one vector per molecule, which is the lossy pooling
# step; "inner" would instead average the density expansion coefficients before the
# power spectrum is built, blurring distinct environments before the invariant exists.
soap = SOAP(species=["H", "C", "N", "O", "S"], r_cut=4.0, n_max=6, l_max=4,
            average="outer", periodic=False)
X_soap = soap.create(subset)
print(f"SOAP: {X_soap.shape[1]} features per molecule (averaged over its atoms), "
      f"{X_soap.shape[0]} molecules")

X_tr, X_te, y_tr, y_te = train_test_split(X_soap, energies, test_size=0.25, random_state=SEED)
forest = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
mae = np.abs(forest.predict(X_te) - y_te).mean()
baseline = np.abs(y_te - y_tr.mean()).mean()
print(f"atomization energy MAE on held-out molecules: {mae:6.1f} kcal/mol")
print(f"predict-the-training-mean baseline:           {baseline:6.1f} kcal/mol")

SOAP has several parameters that interact: the cutoff radius, the radial and angular truncations `n_max` and `l_max`, and the width of the Gaussians placed on each neighbor. The vector grows quickly as they rise, and none of them can be read off the descriptor itself. Choose them the way the modest run above did, then confirm the choice on a held-out score exactly as you would any other modeling decision, rather than trusting a default.

## Manuscript figures


- **This is Figure 2 of the manuscript** (`caffeineCM`): the Coulomb matrix of caffeine, as a heatmap.
- **This is Figure 3 of the manuscript** (`scaling_pca`): PCA of QM7 Coulomb descriptors raw and standardized, with the explained-variance curve that prefers the wrong one.

Two terms in those captions need a definition. A **Coulomb matrix** describes a molecule by
the pairwise quantities $Z_i Z_j / r_{ij}$ between its atoms, with the diagonal encoding each
atom on its own; the off-diagonal entries are what these notebooks use. The **explained
variance ratio** is the share of the total variance each principal component accounts for, so
it tells you how much of the spread the axes capture and nothing about whether that spread is
chemically meaningful.


Each figure is written as **both a PDF and a PNG**, so LaTeX gets vector art and slides get
a raster copy. `scripts/make_manuscript_figures.py` runs this notebook and checks that the
figures appeared, so a cell that quietly skipped one fails the build.

Neither figure carries a plot title, because a published figure carries its title in its
caption. The cell also sizes the type for the printed column, so it changes the shared plot
settings; the last line calls `set_style()` again to put them back, and nothing below this
point is affected.

In [ ]:
# --- Manuscript figures: caffeineCM (Figure 2) and scaling_pca (Figure 3),
# --- both from Section 6 of the article.
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart.

from ase.io import read
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


import os


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


def add_panel_label(ax, label, x=-0.09, y=1.01, size=8.5):
    """Put a bold panel label outside the axes, left of the y-tick labels.

    A panel label is text and not a title, so it stays in the printed figure.
    """
    ax.text(x, y, label, transform=ax.transAxes, fontsize=size,
            fontweight="bold", va="bottom", ha="left", color=SLATE)


def heatmap_cmap():
    """Near-white to purple: one hue, monotone in lightness.

    A single hue reads as a magnitude. A ramp that changes hue as well as
    lightness, viridis included, invites the reader to see categories in what
    is really a continuum.
    """
    return mpl.colors.LinearSegmentedColormap.from_list(
        "manuscript_purple", ["#F6F2FA", LAVENDER, PURPLE])


def scatter_cmap():
    """The same ramp with its palest quarter removed.

    In a filled cell a pale color reads as "low". In a scatter it reads as
    "not there", so the light end has to go before the ramp is used for
    markers. The ordering is unchanged.
    """
    base = heatmap_cmap()
    return mpl.colors.LinearSegmentedColormap.from_list(
        "manuscript_purple_marks", base(np.linspace(0.30, 1.0, 256)))


# Type sized for the printed column rather than for the screen. Each figure is
# authored at exactly the width it is included at, so LaTeX never rescales it
# and a 7.5 pt label is 7.5 pt on the page. The widths are measured from the
# compiled document: \linewidth = 250.95 pt = 3.47 in for one column of the
# two-column layout, \textwidth = 520.40 pt = 7.20 in for both.
COL_W, FULL_W = 3.47, 7.20
PANEL_SIZE = 8.5
mpl.rcParams.update({
    "figure.dpi": 200,
    "font.size": 7.5,
    "axes.labelsize": 7.5,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size": 2.8,
    "ytick.major.size": 2.8,
    "lines.linewidth": 1.3,
})

from rdkit.Chem import AllChem


def caffeine_geometry(seed=SEED):
    """A real caffeine conformer, embedded with ETKDG and relaxed with MMFF.

    The earlier draft carried hand-placed coordinates, which left several atom
    pairs an unphysical one angstrom apart. This takes the same distance-geometry
    route as Section 6.4 instead, so Figure 2 is the Coulomb matrix of an actual
    geometry rather than of a sketch. The atoms are then ordered by element,
    oxygen then nitrogen then carbon then hydrogen, purely so the matrix falls
    into the readable blocks the figure has always shown. That ordering is a
    display choice, not a property of the descriptor: the Coulomb matrix is
    permutation dependent, which is exactly the cautionary point Section 8.6
    turns on, so the same molecule in RDKit's own atom order would draw a
    differently blocked but equally valid heatmap.
    """
    mol = Chem.AddHs(Chem.MolFromSmiles("Cn1cnc2c1c(=O)n(C)c(=O)n2C"))
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    AllChem.EmbedMolecule(mol, params)
    AllChem.MMFFOptimizeMolecule(mol)
    Z = np.array([a.GetAtomicNum() for a in mol.GetAtoms()])
    R = mol.GetConformer().GetPositions()
    order = np.argsort(-Z, kind="stable")
    return R[order], Z[order]


CAFFEINE_R, CAFFEINE_Z = caffeine_geometry()


def coulomb_matrix(R, Z):
    """The full Coulomb matrix of Equation 2 of the manuscript."""
    d = np.linalg.norm(R[:, None] - R[None, :], axis=-1)
    np.fill_diagonal(d, np.inf)
    M = np.outer(Z, Z) / d
    np.fill_diagonal(M, 0.5 * Z.astype(float) ** 2.4)
    return M


def qm7_15atom():
    """Off-diagonal Coulomb descriptors for the QM7 molecules of exactly 15 atoms.

    ASE reads the extended-XYZ file in one call and returns a list of Atoms
    objects. Positions in this particular file are in bohr, not angstrom, which
    changes nothing here because every entry is scaled by the same constant.
    Convert before you quote a bond length or set a cutoff radius.
    """
    frames = [a for a in read(str(DATA / "qm7.xyz"), index=":") if len(a) == 15]
    upper = np.triu_indices(15, k=1)
    X, heaviest = [], []
    for atoms in frames:
        Z = atoms.get_atomic_numbers().astype(float)
        R = atoms.get_positions()
        d = np.linalg.norm(R[:, None] - R[None, :], axis=-1)
        np.fill_diagonal(d, np.inf)
        X.append((np.outer(Z, Z) / d)[upper])
        heaviest.append(Z.max())
    return np.array(X), np.array(heaviest)


def fig_caffeine_coulomb():
    d = np.linalg.norm(CAFFEINE_R[:, None] - CAFFEINE_R[None, :], axis=-1)
    min_sep = d[~np.eye(len(d), dtype=bool)].min()
    assert min_sep > 0.9, (
        f"caffeine conformer has atoms {min_sep:.2f} A apart; the embedding "
        "failed and the Coulomb matrix would blow up"
    )
    print(f"caffeine: {len(CAFFEINE_Z)} atoms, closest pair {min_sep:.2f} A apart")
    M = coulomb_matrix(CAFFEINE_R, CAFFEINE_Z)
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.86))
    im = ax.imshow(M, cmap=heatmap_cmap(), interpolation="nearest")
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("matrix element magnitude", fontsize=7)
    cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_edgecolor(SLATE)
    ax.set_xlabel("atom index")
    ax.set_ylabel("atom index")
    outdir = savefig(fig, "caffeineCM")
    print(f"wrote caffeineCM.png and .pdf to {outdir}")


def fig_qm7_scaling():
    """Scaling changes the PCA, and explained variance prefers the wrong answer.

    Three panels, replacing what were three separate figures in earlier drafts.
    They belong together: (a) and (b) are the same analysis before and after
    standardization, and (c) is the reason a reader who trusted explained
    variance alone would have chosen (a).
    """
    X, heaviest = qm7_15atom()
    Xs = StandardScaler().fit_transform(X)

    # All three panels are drawn the same size and share a baseline. The
    # colorbar gets its own axes instead of being attached to (a) and (b),
    # because attaching it shrinks those two and leaves them visibly smaller
    # than (c).
    fig, axes = plt.subplots(1, 3, figsize=(FULL_W, 2.85))
    fig.subplots_adjust(wspace=0.42, left=0.055, right=0.985, bottom=0.30, top=0.97)
    for ax in axes:
        ax.set_box_aspect(1.0)

    # --- (a) and (b): the same embedding, unscaled then standardized --------
    norm = mpl.colors.Normalize(vmin=heaviest.min(), vmax=heaviest.max())
    # The unscaled / standardized labels that used to sit in these two panels
    # now live in the caption, so which panel is which is stated there.
    for ax, data in zip(axes[:2], [X, Xs]):
        emb = PCA(n_components=2, random_state=SEED).fit_transform(data)
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=heaviest, cmap=scatter_cmap(),
                        norm=norm, s=5, alpha=0.75, linewidths=0)
        ax.set_xlabel("PC 1")
        ax.set_ylabel("PC 2")
        ax.set_xticks([])
        ax.set_yticks([])

    # One colorbar for both embedding panels: the scale is shared, and drawing
    # it twice invites the reader to look for a difference that is not there.
    # It gets its own axes so that (a) and (b) keep the size of (c).
    fig.canvas.draw()
    p0, p1 = axes[0].get_position(), axes[1].get_position()
    cax = fig.add_axes([p0.x0, p0.y0 - 0.155, p1.x1 - p0.x0, 0.035])
    cb = fig.colorbar(sc, cax=cax, orientation="horizontal")
    cb.set_label("atomic number of heaviest atom", fontsize=6.8)
    cb.ax.tick_params(labelsize=6.2)
    cb.outline.set_edgecolor(SLATE)

    # --- (c): cumulative explained variance, both curves on one axes -------
    ax = axes[2]
    for label, data, color, marker, ls in [
        ("standardized", Xs, PALETTE[3], "o", "-"),
        ("unscaled", X, PALETTE[1], "s", "--"),
    ]:
        p = PCA(n_components=30, random_state=SEED).fit(data)
        cum = np.cumsum(p.explained_variance_ratio_)
        ax.plot(np.arange(1, len(cum) + 1), cum, color=color, marker=marker,
                ms=2.6, ls=ls, lw=1.1, label=label)
    ax.axhline(0.9, color=SLATE, ls=":", lw=0.9)
    ax.text(1.0, 0.915, "90%", ha="left", va="bottom", fontsize=6.2, color=SLATE)
    ax.set_ylim(0, 1.03)
    ax.set_xlabel("principal components")
    ax.set_ylabel("cumulative explained variance")
    ax.legend(loc="lower right", fontsize=6.4, frameon=False)

    # Panel (c) carries a long y-axis label, so its marker needs more clearance
    # than the two embedding panels to sit level with them on the page.
    for ax, lab, dx in zip(axes, ["(a)", "(b)", "(c)"], [-0.13, -0.13, -0.19]):
        add_panel_label(ax, lab, x=dx, y=1.02, size=PANEL_SIZE)

    outdir = savefig(fig, "scaling_pca")
    print(f"wrote scaling_pca.png and .pdf to {outdir}")


fig_caffeine_coulomb()
fig_qm7_scaling()
plt.show()

# Put the shared plot settings back, so anything you add below this point gets
# screen-sized type again.
set_style()

### Exercise

Add a third representation to the neighbor-overlap comparison in section 3: MACCS keys with
Euclidean distance instead of Jaccard. Does changing only the *metric* move the neighbor sets as
much as changing the *representation* did?

In [ ]:
# YOUR CODE HERE